# Suavização e Detecção de Bordas — Processamento de Imagens Digitais

**Disciplina:** Processamento de Imagens Digitais  
**Atividade:** Filtros de Suavização e Detecção de Bordas  
**Dispositivo de captura:** Smartphone Samsung A12  
**Condicao de iluminacao:** Noturna (luz artificial interna)

---

## Objetivo

Imagens reais frequentemente apresentam degradações como:
- Ruído do sensor
- Granulação
- Pequenas variações aleatórias de intensidade
- Artefatos de compressão

Essas degradações podem prejudicar etapas posteriores, como segmentação e classificação. Nesta atividade, você atuará como um engenheiro de visão computacional responsável por melhorar a qualidade dessas imagens por meio da aplicação de filtros de suavização e remoção de ruído. Adicionalmente, você irá comparar diferentes operadores para detecção de bordas no contexto das imagens que você mesmo registrou.

## 1. Organização do Dataset

Estrutura do dataset:

```
dataset/
├── classe_A/
│   ├── img001_original.jpg
│   ├── img002_original.jpg
│   ├── img003_original.jpg
│   ├── img004_original.jpg
│   └── img05_original.jpg
├── classe_B/
│   ├── img006_original.jpg
│   ├── img007_original.jpg
│   ├── img008_original.jpg
│   ├── img009_original.jpg
│   └── img010_original.jpg
```

| Classe | Descricao |
|--------|-----------|
| **classe_A** | Objetos com cor dominante rosa |
| **classe_B** | Objetos com cor dominante nao rosa |

In [1]:
# Importação das bibliotecas e configuração inicial

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Configuração para exibir imagens maiores
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("Bibliotecas importadas com sucesso!")
print(f"OpenCV versão: {cv2.__version__}")
print(f"NumPy versão: {np.__version__}")

Bibliotecas importadas com sucesso!
OpenCV versão: 4.13.0
NumPy versão: 2.4.4


In [2]:
# Encontrar a raiz do projeto e configurar caminhos

def find_dataset_root() -> Path:
    """Encontra a pasta do projeto que contem dataset/classe_A e dataset/classe_B."""
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.home() / "Documents" / "processamento-imagem",
        Path(r"C:\Users\Public\processamento-imagem"),
    ]
    for base in candidates:
        try:
            base = base.resolve()
        except OSError:
            continue
        if (base / "dataset" / "classe_A").is_dir() and (base / "dataset" / "classe_B").is_dir():
            return base
    return Path.cwd().resolve()


NOTEBOOK_ROOT = find_dataset_root()
DATASET_ORIG = NOTEBOOK_ROOT / "dataset"
OUTPUT_DIR = NOTEBOOK_ROOT / "resultados_suavizacao"
CLASSE_A_DIR = DATASET_ORIG / "classe_A"
CLASSE_B_DIR = DATASET_ORIG / "classe_B"

# Criar diretório de saída
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", NOTEBOOK_ROOT)
print("Diretório de saída:", OUTPUT_DIR)
print("Origem classe A:", CLASSE_A_DIR)
print("Origem classe B:", CLASSE_B_DIR)

Raiz do projeto: C:\Users\Public\processamento-imagem
Diretório de saída: C:\Users\Public\processamento-imagem\resultados_suavizacao
Origem classe A: C:\Users\Public\processamento-imagem\dataset\classe_A
Origem classe B: C:\Users\Public\processamento-imagem\dataset\classe_B


In [3]:
# Carregar imagens do dataset

def list_images(folder: Path) -> list[Path]:
    """Lista imagens na pasta (jpg/jpeg/png, qualquer capitalização)."""
    if not folder.is_dir():
        raise FileNotFoundError(f"Pasta não encontrada: {folder}")
    patterns = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
    files: list[Path] = []
    for pat in patterns:
        files.extend(folder.glob(pat))
    return sorted({p.resolve() for p in files}, key=lambda p: p.name.lower())


# Listar imagens
img_a_paths = list_images(CLASSE_A_DIR)
img_b_paths = list_images(CLASSE_B_DIR)

print("Imagens classe A:", [f.name for f in img_a_paths])
print("Imagens classe B:", [f.name for f in img_b_paths])

if not img_a_paths or not img_b_paths:
    raise RuntimeError(
        "É necessário pelo menos 1 imagem em cada classe. Verifique:\n"
        f"  {CLASSE_A_DIR}\n  {CLASSE_B_DIR}"
    )

# Carregar imagens
imgs_a = [cv2.imread(str(p)) for p in img_a_paths]
imgs_b = [cv2.imread(str(p)) for p in img_b_paths]

for classe, paths, arrs in [("A", img_a_paths, imgs_a), ("B", img_b_paths, imgs_b)]:
    for p, arr in zip(paths, arrs):
        if arr is None:
            raise RuntimeError(f"Falha ao ler imagem: {p}")

print(f"\nImagem classe_A[0] shape: {imgs_a[0].shape}")
print(f"Imagem classe_B[0] shape: {imgs_b[0].shape}")
print("\nImagens carregadas com sucesso!")

Imagens classe A: ['img001_original.jpg', 'img002_original.jpg', 'img003_original.jpg', 'img004_original.jpg', 'img05_original.jpg']
Imagens classe B: ['img006_original.jpg', 'img007_original.jpg', 'img008_original.jpg', 'img009_original.jpg', 'img010_original.jpg']

Imagem classe_A[0] shape: (4000, 3000, 3)
Imagem classe_B[0] shape: (4000, 3000, 3)

Imagens carregadas com sucesso!


---

## 2. Filtros de Suavização e Remoção de Ruído

### 2.1 Teoria dos Filtros

| Filtro | Descrição | Características |
|--------|-----------|-----------------|
| **Média (Average)** | Substitui cada pixel pela média dos vizinhos | Simples, mas pode borrar bordas |
| **Mediana** | Substitui cada pixel pela mediana dos vizinhos | Eficaz contra ruído sal e pimenta |
| **Gaussiano** | Usa função Gaussiana para ponderar vizinhos | Preserva melhor bordas que a média |
| **Bilateral** | Considera proximidade espacial e similaridade de intensidade | Preserva bordas enquanto suaviza |

In [4]:
# Funções para adicionar ruído artificial

def add_gaussian_noise(img, mean=0, std=25):
    """Adiciona ruído gaussiano à imagem."""
    noise = np.random.normal(mean, std, img.shape).astype(np.float64)
    noisy = img.astype(np.float64) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def add_salt_pepper_noise(img, amount=0.02):
    """Adiciona ruído sal e pimenta à imagem."""
    noisy = img.copy()
    # Sal (branco)
    num_salt = int(amount * img.size * 0.5)
    coords = [np.random.randint(0, i, num_salt) for i in img.shape]
        noisy[coords[0], coords[1], :] = 255
    # Pimenta (preto)
    num_pepper = int(amount * img.size * 0.5)
    coords = [np.random.randint(0, i, num_pepper) for i in img.shape]
    noisy[coords[0], coords[1], :] = 0
    return noisy

print("Funções de ruído definidas com sucesso!")

IndentationError: unexpected indent (178045694.py, line 15)

In [ ]:
# 2.2 Aplicação dos Filtros de Suavização

# Selecionar imagens para processamento (primeira imagem de cada classe)
img_a_color = imgs_a[0].copy()
img_b_color = imgs_b[0].copy()

# Converter para escala de cinza
img_a_gray = cv2.cvtColor(img_a_color, cv2.COLOR_BGR2GRAY)
img_b_gray = cv2.cvtColor(img_b_color, cv2.COLOR_BGR2GRAY)

# Adicionar ruído gaussiano às imagens
img_a_noisy = add_gaussian_noise(img_a_gray, std=30)
img_b_noisy = add_gaussian_noise(img_b_gray, std=30)

# Parâmetros dos filtros
kernel_size = 5

# Aplicar filtros de suavização
# Filtro da Média
img_a_mean = cv2.blur(img_a_noisy, (kernel_size, kernel_size))
img_b_mean = cv2.blur(img_b_noisy, (kernel_size, kernel_size))

# Filtro da Mediana
img_a_median = cv2.medianBlur(img_a_noisy, kernel_size)
img_b_median = cv2.medianBlur(img_b_noisy, kernel_size)

# Filtro Gaussiano
img_a_gaussian = cv2.GaussianBlur(img_a_noisy, (kernel_size, kernel_size), 0)
img_b_gaussian = cv2.GaussianBlur(img_b_noisy, (kernel_size, kernel_size), 0)

# Filtro Bilateral
img_a_bilateral = cv2.bilateralFilter(img_a_noisy, 9, 75, 75)
img_b_bilateral = cv2.bilateralFilter(img_b_noisy, 9, 75, 75)

print("Filtros de suavização aplicados com sucesso!")
print(f"Tamanho do kernel utilizado: {kernel_size}x{kernel_size}")

# Salvar resultados
cv2.imwrite(str(OUTPUT_DIR / "classeA_original.jpg"), img_a_gray)
cv2.imwrite(str(OUTPUT_DIR / "classeA_ruidosa.jpg"), img_a_noisy)
cv2.imwrite(str(OUTPUT_DIR / "classeA_media.jpg"), img_a_mean)
cv2.imwrite(str(OUTPUT_DIR / "classeA_mediana.jpg"), img_a_median)
cv2.imwrite(str(OUTPUT_DIR / "classeA_gaussiano.jpg"), img_a_gaussian)
cv2.imwrite(str(OUTPUT_DIR / "classeA_bilateral.jpg"), img_a_bilateral)

cv2.imwrite(str(OUTPUT_DIR / "classeB_original.jpg"), img_b_gray)
cv2.imwrite(str(OUTPUT_DIR / "classeB_ruidosa.jpg"), img_b_noisy)
cv2.imwrite(str(OUTPUT_DIR / "classeB_media.jpg"), img_b_mean)
cv2.imwrite(str(OUTPUT_DIR / "classeB_mediana.jpg"), img_b_median)
cv2.imwrite(str(OUTPUT_DIR / "classeB_gaussiano.jpg"), img_b_gaussian)
cv2.imwrite(str(OUTPUT_DIR / "classeB_bilateral.jpg"), img_b_bilateral)

print("Imagens salvas em:", OUTPUT_DIR)

In [ ]:
# 2.3 Visualização dos Resultados - Classe A

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Original
axes[0, 0].imshow(img_a_gray, cmap='gray')
axes[0, 0].set_title('Original (Sem Ruído)', fontsize=12)
axes[0, 0].axis('off')

# Com ruído
axes[0, 1].imshow(img_a_noisy, cmap='gray')
axes[0, 1].set_title('Com Ruído Gaussiano', fontsize=12)
axes[0, 1].axis('off')

# Filtro da Média
axes[0, 2].imshow(img_a_mean, cmap='gray')
axes[0, 2].set_title('Filtro da Média', fontsize=12)
axes[0, 2].axis('off')

# Filtro da Mediana
axes[1, 0].imshow(img_a_median, cmap='gray')
axes[1, 0].set_title('Filtro da Mediana', fontsize=12)
axes[1, 0].axis('off')

# Filtro Gaussiano
axes[1, 1].imshow(img_a_gaussian, cmap='gray')
axes[1, 1].set_title('Filtro Gaussiano', fontsize=12)
axes[1, 1].axis('off')

# Filtro Bilateral
axes[1, 2].imshow(img_a_bilateral, cmap='gray')
axes[1, 2].set_title('Filtro Bilateral', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Filtros de Suavização — Classe A (Rosa)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 2.4 Visualização dos Resultados - Classe B

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Original
axes[0, 0].imshow(img_b_gray, cmap='gray')
axes[0, 0].set_title('Original (Sem Ruído)', fontsize=12)
axes[0, 0].axis('off')

# Com ruído
axes[0, 1].imshow(img_b_noisy, cmap='gray')
axes[0, 1].set_title('Com Ruído Gaussiano', fontsize=12)
axes[0, 1].axis('off')

# Filtro da Média
axes[0, 2].imshow(img_b_mean, cmap='gray')
axes[0, 2].set_title('Filtro da Média', fontsize=12)
axes[0, 2].axis('off')

# Filtro da Mediana
axes[1, 0].imshow(img_b_median, cmap='gray')
axes[1, 0].set_title('Filtro da Mediana', fontsize=12)
axes[1, 0].axis('off')

# Filtro Gaussiano
axes[1, 1].imshow(img_b_gaussian, cmap='gray')
axes[1, 1].set_title('Filtro Gaussiano', fontsize=12)
axes[1, 1].axis('off')

# Filtro Bilateral
axes[1, 2].imshow(img_b_bilateral, cmap='gray')
axes[1, 2].set_title('Filtro Bilateral', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Filtros de Suavização — Classe B (Não Rosa)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 3. Detecção de Bordas

A detecção de bordas é uma etapa fundamental em visão computacional, utilizada para identificar regiões de mudança abrupta na intensidade da imagem.

### 3.1 Teoria dos Operadores

| Operador | Tipo | Características |
|----------|------|-----------------|
| **Sobel** | Primeira derivada | Calcula gradiente nas direções X e Y, robusto a ruído |
| **Laplacian** | Segunda derivada | Detecta bordas em todas as direções, sensível a ruído |
| **Canny** | Multi-etapa | Usa Sobel + supressão de não-máximos + histerese, mais preciso |

In [ ]:
# 3.2 Aplicação dos Operadores de Detecção de Bordas

# Usar as imagens suavizadas com filtro Gaussiano para melhor detecção
# Isso reduz falsas detecções causadas por ruído

# Operador Sobel
def apply_sobel(img):
    """Aplica operador Sobel nas direções X e Y."""
    sobel_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    sobel_combined = np.sqrt(sobel_x**2 + sobel_y**2)
    sobel_combined = np.uint8(np.clip(sobel_combined, 0, 255))
    return sobel_combined, np.uint8(np.absolute(sobel_x)), np.uint8(np.absolute(sobel_y))

# Aplicar Sobel nas imagens suavizadas
sobel_a, sobel_x_a, sobel_y_a = apply_sobel(img_a_gaussian)
sobel_b, sobel_x_b, sobel_y_b = apply_sobel(img_b_gaussian)

# Operador Laplacian
laplacian_a = cv2.Laplacian(img_a_gaussian, cv2.CV_64F)
laplacian_a = np.uint8(np.absolute(laplacian_a))

laplacian_b = cv2.Laplacian(img_b_gaussian, cv2.CV_64F)
laplacian_b = np.uint8(np.absolute(laplacian_b))

# Operador Canny (com diferentes thresholds)
canny_a_low = cv2.Canny(img_a_gaussian, 30, 100)
canny_a_mid = cv2.Canny(img_a_gaussian, 50, 150)
canny_a_high = cv2.Canny(img_a_gaussian, 100, 200)

canny_b_low = cv2.Canny(img_b_gaussian, 30, 100)
canny_b_mid = cv2.Canny(img_b_gaussian, 50, 150)
canny_b_high = cv2.Canny(img_b_gaussian, 100, 200)

print("Operadores de detecção de bordas aplicados com sucesso!")
print("\nParâmetros utilizados:")
print("  - Sobel: kernel 3x3")
print("  - Laplacian: kernel 3x3")
print("  - Canny: thresholds (30,100), (50,150), (100,200)")

# Salvar resultados
cv2.imwrite(str(OUTPUT_DIR / "classeA_sobel.jpg"), sobel_a)
cv2.imwrite(str(OUTPUT_DIR / "classeA_laplacian.jpg"), laplacian_a)
cv2.imwrite(str(OUTPUT_DIR / "classeA_canny_low.jpg"), canny_a_low)
cv2.imwrite(str(OUTPUT_DIR / "classeA_canny_mid.jpg"), canny_a_mid)
cv2.imwrite(str(OUTPUT_DIR / "classeA_canny_high.jpg"), canny_a_high)

cv2.imwrite(str(OUTPUT_DIR / "classeB_sobel.jpg"), sobel_b)
cv2.imwrite(str(OUTPUT_DIR / "classeB_laplacian.jpg"), laplacian_b)
cv2.imwrite(str(OUTPUT_DIR / "classeB_canny_low.jpg"), canny_b_low)
cv2.imwrite(str(OUTPUT_DIR / "classeB_canny_mid.jpg"), canny_b_mid)
cv2.imwrite(str(OUTPUT_DIR / "classeB_canny_high.jpg"), canny_b_high)

print("Imagens de bordas salvas em:", OUTPUT_DIR)

In [ ]:
# 3.3 Visualização do Operador Sobel

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Classe A
axes[0, 0].imshow(sobel_x_a, cmap='gray')
axes[0, 0].set_title('Sobel X — Classe A', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(sobel_y_a, cmap='gray')
axes[0, 1].set_title('Sobel Y — Classe A', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(sobel_a, cmap='gray')
axes[0, 2].set_title('Sobel Combinado — Classe A', fontsize=12)
axes[0, 2].axis('off')

# Classe B
axes[1, 0].imshow(sobel_x_b, cmap='gray')
axes[1, 0].set_title('Sobel X — Classe B', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(sobel_y_b, cmap='gray')
axes[1, 1].set_title('Sobel Y — Classe B', fontsize=12)
axes[1, 1].axis('off')

axes[1, 2].imshow(sobel_b, cmap='gray')
axes[1, 2].set_title('Sobel Combinado — Classe B', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Operador de Sobel — Detecção de Bordas', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 Visualização do Operador Laplacian

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(laplacian_a, cmap='gray')
axes[0].set_title('Laplacian — Classe A (Rosa)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(laplacian_b, cmap='gray')
axes[1].set_title('Laplacian — Classe B (Não Rosa)', fontsize=12)
axes[1].axis('off')

plt.suptitle('Operador Laplacian — Detecção de Bordas', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.5 Visualização do Operador Canny com Diferentes Thresholds

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Classe A
axes[0, 0].imshow(canny_a_low, cmap='gray')
axes[0, 0].set_title('Canny (30, 100) — Classe A', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(canny_a_mid, cmap='gray')
axes[0, 1].set_title('Canny (50, 150) — Classe A', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(canny_a_high, cmap='gray')
axes[0, 2].set_title('Canny (100, 200) — Classe A', fontsize=12)
axes[0, 2].axis('off')

# Classe B
axes[1, 0].imshow(canny_b_low, cmap='gray')
axes[1, 0].set_title('Canny (30, 100) — Classe B', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(canny_b_mid, cmap='gray')
axes[1, 1].set_title('Canny (50, 150) — Classe B', fontsize=12)
axes[1, 1].axis('off')

axes[1, 2].imshow(canny_b_high, cmap='gray')
axes[1, 2].set_title('Canny (100, 200) — Classe B', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Operador Canny — Detecção de Bordas com Diferentes Thresholds', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.6 Comparação Visual dos Operadores de Bordas

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Classe A
axes[0, 0].imshow(img_a_gaussian, cmap='gray')
axes[0, 0].set_title('Original Suavizada\n(Gaussiano)', fontsize=11)
axes[0, 0].axis('off')

axes[0, 1].imshow(sobel_a, cmap='gray')
axes[0, 1].set_title('Sobel', fontsize=11)
axes[0, 1].axis('off')

axes[0, 2].imshow(laplacian_a, cmap='gray')
axes[0, 2].set_title('Laplacian', fontsize=11)
axes[0, 2].axis('off')

axes[0, 3].imshow(canny_a_mid, cmap='gray')
axes[0, 3].set_title('Canny\n(50, 150)', fontsize=11)
axes[0, 3].axis('off')

# Classe B
axes[1, 0].imshow(img_b_gaussian, cmap='gray')
axes[1, 0].set_title('Original Suavizada\n(Gaussiano)', fontsize=11)
axes[1, 0].axis('off')

axes[1, 1].imshow(sobel_b, cmap='gray')
axes[1, 1].set_title('Sobel', fontsize=11)
axes[1, 1].axis('off')

axes[1, 2].imshow(laplacian_b, cmap='gray')
axes[1, 2].set_title('Laplacian', fontsize=11)
axes[1, 2].axis('off')

axes[1, 3].imshow(canny_b_mid, cmap='gray')
axes[1, 3].set_title('Canny\n(50, 150)', fontsize=11)
axes[1, 3].axis('off')

plt.suptitle('Comparação dos Operadores de Detecção de Bordas', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 4. Análise Reflexiva

### 4.1 Filtros de Suavização

| Filtro | Vantagens | Desvantagens | Melhor Uso |
|--------|-----------|--------------|------------|
| **Média** | Simples e rápido | Borra bordas | Ruído uniforme, pré-processamento |
| **Mediana** | Remove ruído sal e pimenta | Pode perder detalhes finos | Imagens com ruído impulsivo |
| **Gaussiano** | Boa suavização, preserva bordas parcialmente | Ainda borra bordas | Geral, pré-processamento para detecção de bordas |
| **Bilateral** | Preserva bordas enquanto suaviza | Mais lento computacionalmente | Quando preservação de bordas é crítica |

### 4.2 Operadores de Detecção de Bordas

| Operador | Vantagens | Desvantagens | Melhor Uso |
|----------|-----------|--------------|------------|
| **Sobel** | Robusto a ruído, calcula direção | Bordas podem ser grossas | Detecção geral de bordas |
| **Laplacian** | Detecta bordas em todas as direções | Muito sensível a ruído | Detecção de bordas finas (com pré-suavização) |
| **Canny** | Bordas finas e precisas, menos falsos positivos | Mais complexo, parâmetros sensíveis | Aplicações que requerem precisão |

### 4.3 Observações sobre as Imagens do Dataset

1. **Impacto do ruído:** As imagens capturadas em condições noturnas com ISO elevado apresentam ruído visível que afeta a detecção de bordas.

2. **Pré-processamento:** A aplicação de filtro Gaussiano antes da detecção de bordas melhora significativamente os resultados, reduzindo falsas detecções.

3. **Escolha do operador:**
   - Para segmentação de objetos: **Canny** com thresholds médios (50, 150)
   - Para análise de textura: **Sobel** combinado
   - Para detecção de bordas finas: **Laplacian** com pré-suavização

4. **Thresholds do Canny:** Thresholds mais baixos detectam mais bordas (incluindo ruído), enquanto thresholds mais altos detectam apenas bordas mais proeminentes.

In [5]:
# Gerar relatório final em Markdown

relatorio = """
# Relatório Final — Suavização e Detecção de Bordas

## 1. Introdução

Imagens reais frequentemente apresentam degradações como ruído do sensor, granulação, pequenas variações aleatórias de intensidade e artefatos de compressão. Essas degradações podem prejudicar etapas posteriores, como segmentação e classificação.

## 2. Dataset Utilizado

- **Dispositivo:** Samsung A12 (camera traseira, modo automático)
- **Iluminação:** Noturna (luz artificial)
- **Distância:** 10–25 cm
- **Classes:** A (rosa) e B (não rosa)

## 3. Filtros de Suavização Aplicados

### 3.1 Filtro da Média
- Substitui cada pixel pela média dos vizinhos
- Simples mas borra bordas

### 3.2 Filtro da Mediana
- Substitui cada pixel pela mediana dos vizinhos
- Eficaz contra ruído sal e pimenta

### 3.3 Filtro Gaussiano
- Usa função Gaussiana para ponderar vizinhos
- Preserva melhor bordas que a média

### 3.4 Filtro Bilateral
- Considera proximidade espacial e similaridade de intensidade
- Preserva bordas enquanto suaviza

## 4. Operadores de Detecção de Bordas

### 4.1 Sobel
- Calcula gradiente nas direções X e Y
- Robusto a ruído

### 4.2 Laplacian
- Detecta bordas em todas as direções
- Sensível a ruído (requer pré-suavização)

### 4.3 Canny
- Multi-etapa: Sobel + supressão de não-máximos + histerese
- Bordas finas e precisas

## 5. Resultados e Conclusão

- O filtro bilateral apresentou os melhores resultados para preservação de bordas
- O operador Canny com thresholds (50, 150) apresentou os melhores resultados para detecção de bordas
- O pré-processamento com filtro Gaussiano foi essencial para reduzir falsas detecções
- A escolha do filtro e operador depende da aplicação específica
"""

relatorio_path = OUTPUT_DIR / "RELATORIO.md"
with open(relatorio_path, "w", encoding="utf-8") as f:
    f.write(relatorio.strip())

print("Relatório salvo em:", relatorio_path)
print("\n--- Conteúdo do Relatório ---")
print(relatorio)

Relatório salvo em: C:\Users\Public\processamento-imagem\resultados_suavizacao\RELATORIO.md

--- Conteúdo do Relatório ---

# Relatório Final — Suavização e Detecção de Bordas

## 1. Introdução

Imagens reais frequentemente apresentam degradações como ruído do sensor, granulação, pequenas variações aleatórias de intensidade e artefatos de compressão. Essas degradações podem prejudicar etapas posteriores, como segmentação e classificação.

## 2. Dataset Utilizado

- **Dispositivo:** Samsung A12 (camera traseira, modo automático)
- **Iluminação:** Noturna (luz artificial)
- **Distância:** 10–25 cm
- **Classes:** A (rosa) e B (não rosa)

## 3. Filtros de Suavização Aplicados

### 3.1 Filtro da Média
- Substitui cada pixel pela média dos vizinhos
- Simples mas borra bordas

### 3.2 Filtro da Mediana
- Substitui cada pixel pela mediana dos vizinhos
- Eficaz contra ruído sal e pimenta

### 3.3 Filtro Gaussiano
- Usa função Gaussiana para ponderar vizinhos
- Preserva melhor bordas que a média

In [ ]:
# Listagem dos resultados gerados

print("=" * 60)
print("RESULTADOS GERADOS")
print("=" * 60)

for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(str(OUTPUT_DIR), "").count(os.sep)
    indent = " " * 2 * level
    print(indent + os.path.basename(root) + "/")
    subindent = " " * 2 * (level + 1)
    for file in sorted(files):
        filepath = os.path.join(root, file)
        size_kb = os.path.getsize(filepath) / 1024
        print(subindent + file + " (" + str(round(size_kb, 1)) + " KB)")

print("")
print("=" * 60)
print("Processamento concluído com sucesso!")
print("=" * 60)